# Sentiment Analysis

**Sentiment analysis** estimates whether a piece of writing expresses positive, negative or neutral language. It can help us explore patterns in reviews, captions, comments and social-media posts. In *Cartographies of Affect*, we might compare descriptions of public spaces or examine how places are represented through text.

**An important limitation:** sentiment scores describe patterns in **language**, not the actual emotions of a person or the quality of a place. Words can be ironic, ambiguous, culturally specific or dependent on context. We should not treat the scores as objective measurements of urban experience.

In this notebook we compare two lightweight methods, **TextBlob** and **VADER**, then apply sentiment analysis to a small Pandas DataFrame. At the end, an optional example introduces a more recent approach based on a pretrained **transformer language model**.

## TextBlob

![TextBlob logo](https://diamond-thumbnails.s3.us-west-2.amazonaws.com/thinkbigcms/Product/logo/0643d8c9-2d6f-4305-8733-cd96b8557578.png?hash=350304961fe86c1de39bf5fbc025f1ea)

TextBlob offers several tools for analysing text. Its **default sentiment analyser is lexicon- and rule-based**, using words and phrases associated with sentiment. It returns two scores:

- **Polarity**, from **−1 to +1**: negative, neutral or positive.
- **Subjectivity**, from **0 to 1**: language estimated to be more factual or more opinion-based.

![Diagram showing polarity and subjectivity scales](assets/sentiment-scales.svg)

Neither score is a probability that the writer actually felt a particular emotion.

[TextBlob documentation](https://textblob.readthedocs.io/en/dev/quickstart.html).

We begin with the existing example: a strongly negative film review of *Jaws: The Revenge* by Gene Siskel. The longer passage is useful for seeing how TextBlob produces a single score from several sentences.

In [1]:
from textblob import TextBlob

text_to_analyse = """In the just-released 'Jaws: The Revenge' the shark's main
course is intended to be Roy Scheider's widow, Ellen Brody, a frumpy middle-aged
woman played by boring actress Lorraine Gary, who happens to be married to the
president of MCA Universal, which finances the 'Jaws' films and which explains
her lead role. Let's put it this way: When you see and hear the nasal
Lorraine Gary on screen you want the shark to eat her."""

text = TextBlob(text_to_analyse)

print("Polarity: " + str(text.sentiment.polarity))
print("Subjectivity: " + str(text.sentiment.subjectivity))


Polarity: -0.14583333333333334
Subjectivity: 0.3958333333333333


### Interpreting the TextBlob results

**Polarity** gives an overall estimate of negative or positive language; **subjectivity** estimates how opinion-like the wording is. A subjectivity score of 1 is not proof that the review is false, and a score of 0 does not prove that it is factual.

The review contains strongly negative descriptions, so we should expect a negative polarity. But the result depends on TextBlob's dictionary and rules; sarcasm or context may be missed.

A useful comparison is to try a shorter, neutral sentence and inspect both values:

In [ ]:
neutral_text = TextBlob("The photograph was taken near the station.")

print("Polarity:", neutral_text.sentiment.polarity)
print("Subjectivity:", neutral_text.sentiment.subjectivity)

## VADER (Valence Aware Dictionary and sEntiment Reasoner)

VADER is another **lexicon- and rule-based** method, designed especially for short, informal messages such as social-media posts. It takes into account some punctuation, capitalisation, intensifiers and emoticons. Like TextBlob, it does not need training on our dataset.

For each text, VADER returns a dictionary containing four scores:

| Score | Meaning |
| --- | --- |
| `neg` | Relative proportion of negative sentiment |
| `neu` | Relative proportion of neutral sentiment |
| `pos` | Relative proportion of positive sentiment |
| `compound` | Combined, normalised sentiment score from **−1 to +1** |

The three proportion scores sum to approximately **1**. The `compound` score is not a probability. We can use it as a simple indicator when comparing short texts.

VADER is quick and transparent, but can still misread sarcasm, nuanced language and subjects outside its original social-media context.

[Official VADER documentation](https://vadersentiment.readthedocs.io/en/latest/) · [Project repository](https://github.com/cjhutto/vaderSentiment).

In [5]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


analyzer = SentimentIntensityAnalyzer()
examples = [
    "I love this! It's amazing and makes me so happy.",
    "This is the worst. I hate it and I'm disappointed.",
    "I'm not sure how I feel about this...",
    "LOL that was hilarious! :D",
    "The product is okay, but not great."
]

for text in examples:
    scores = analyzer.polarity_scores(text)
    print(text)
    print('  ->', scores)

I love this! It's amazing and makes me so happy.
  -> {'neg': 0.0, 'neu': 0.349, 'pos': 0.651, 'compound': 0.9329}
This is the worst. I hate it and I'm disappointed.
  -> {'neg': 0.609, 'neu': 0.391, 'pos': 0.0, 'compound': -0.8979}
I'm not sure how I feel about this...
  -> {'neg': 0.219, 'neu': 0.781, 'pos': 0.0, 'compound': -0.2411}
LOL that was hilarious! :D
  -> {'neg': 0.0, 'neu': 0.159, 'pos': 0.841, 'compound': 0.89}
The product is okay, but not great.
  -> {'neg': 0.408, 'neu': 0.459, 'pos': 0.133, 'compound': -0.6112}


## Analysing several texts with Pandas

Until now, we have analysed one text at a time. Pandas allows us to apply the same operation to **every row of a table**. This connects sentiment analysis to the dataset workflow from the previous tutorials.

The following comments are illustrative, not collected from social media. We will create a DataFrame, calculate VADER's `compound` score for each text, and store it in a new column.

In [ ]:
import pandas as pd

comments = [
    "I love this neighbourhood!",
    "The streets are noisy and unpleasant.",
    "There is a park near the station.",
    "What a wonderful public square!"
]

df = pd.DataFrame({"comment": comments})

def sentiment_score(text):
    return analyzer.polarity_scores(text)["compound"]

df["sentiment"] = df["comment"].apply(sentiment_score)

df

The method `.apply()` runs our `sentiment_score()` function on every value in the `comment` column. We could use the same method with Flickr titles or descriptions after checking for missing text.

**Interpreting the results:** Compare the scores with the original statements. A neutral-looking sentence may still be classified positively or negatively, and a single score does not tell us what people experience in a space.

## A more recent approach: transformer models (optional)

TextBlob and VADER are useful for learning because they are relatively simple and quick to run. Today, sentiment classification can also be performed with pretrained **transformer models**. These models learn patterns from large text datasets and can recognise more context than a fixed sentiment dictionary, although they also make mistakes and carry biases from their training data.

The [Hugging Face Transformers library](https://huggingface.co/docs/transformers/main_classes/pipelines) provides a `pipeline()` interface. Here we use [Cardiff NLP's Twitter-RoBERTa sentiment model](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest), designed for **English social-media text**. It predicts **negative, neutral or positive**, along with a confidence-like model score. That score is **not directly comparable** to the −1 to +1 polarity scores from TextBlob or VADER.

This final example is optional: the model downloads its weights on first use, requires additional Python packages and may take more time than the previous examples. It is not necessary for the core workshop.

**Setup:** Run the following command in a notebook code cell **if you want to try the optional model**:

```python
%pip install transformers torch
```

After installation, a kernel restart may be required, depending on the environment.

In [ ]:
# Optional: set to True after installing transformers and torch
RUN_TRANSFORMER_EXAMPLE = False

if RUN_TRANSFORMER_EXAMPLE:
    from transformers import pipeline

    modern_classifier = pipeline(
        "text-classification",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest"
    )

    results = modern_classifier([
        "I love this neighbourhood!",
        "The streets are noisy and unpleasant.",
        "There is a park near the station."
    ])

    print(results)

### What should we use?

There is no single model that is automatically best for every dataset. **VADER** is a useful baseline for short English-language comments; **TextBlob** is a simple introduction to polarity and subjectivity; a **transformer** may capture more context, but needs more resources and still requires validation.

Before applying any method to urban research, inspect a sample of the original texts yourself. Note the language, subject matter, geographic context, mistakes and uncertain cases. In particular, a classifier trained on social-media posts should not automatically be trusted to evaluate descriptions of architecture or public space.

For multilingual material, a model specifically trained for the relevant languages may be more appropriate than an English-only model. [Cardiff NLP's multilingual sentiment models](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual) provide one starting point for further exploration.